# Phase 8 — Resume Quality Analyzer

## Objective

This notebook implements the Resume Quality Analyzer component of SkillSync AI.

The purpose of this component is to evaluate the overall quality and completeness of a candidate's resume.

The Resume Quality Analyzer works independently from the Resume ↔ Job Description Matching and Skill Gap Analysis components.

While the matching system determines how well a resume matches a specific Job Description, the Resume Quality Analyzer evaluates whether the resume itself contains important information and has sufficient content for further analysis.

## Input

The analyzer will use the cleaned resume text produced during the earlier preprocessing phase.

The main input is:

- Candidate resume text

## Quality Analysis

The analyzer will examine measurable characteristics of the resume, such as:

- Resume length
- Presence of contact information
- Presence of education-related information
- Presence of experience-related information
- Presence of skills
- Presence of common resume sections
- Overall content completeness

## Process

Cleaned Resume
↓
Resume Quality Analysis
↓
Content and Section Checks
↓
Quality Indicators
↓
Quality Summary

## Expected Output

The Resume Quality Analyzer will provide:

1. Resume length information
2. Detected resume sections
3. Missing or weak sections
4. Content completeness indicators
5. Quality-related observations

## Important Design Principle

The Resume Quality Analyzer should provide measurable and explainable results.

The system should not claim that a resume is "good" or "bad" based on an arbitrary score.

Instead, it will identify specific characteristics of the resume and report the results clearly.

This makes the analysis easier to understand and integrate into the final SkillSync AI application.

## Relationship with Other Components

Resume Quality Analyzer:

Resume
↓
Quality and Completeness Analysis

Resume ↔ JD Matching:

Resume + Job Description
↓
Skill Matching
↓
Skill Gaps

Both components provide different types of information and will later be combined by the SkillSync AI backend.

## Step 1 — Load Required Libraries

In [1]:
import re
import pandas as pd

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 2 — Load Cleaned Resume Dataset

The Resume Quality Analyzer will use the cleaned resume dataset created during Phase 2.

Using the cleaned resume text ensures that the quality analysis is performed on normalized text rather than the original unprocessed resume content.

In [2]:
resume_df = pd.read_csv(
    "../data/processed/resumes_clean.csv"
)

print("Resume dataset loaded successfully.")
print("Number of resumes:", len(resume_df))
print("\nColumns:")
print(resume_df.columns.tolist())

Resume dataset loaded successfully.
Number of resumes: 2484

Columns:
['ID', 'Resume_str', 'Resume_html', 'Category', 'text_length', 'html_length', 'clean_resume', 'raw_length', 'clean_length', 'retention_ratio']


## Step 3 — Inspect Resume Content

Before creating quality rules, the resume text should be inspected to understand the information available to the analyzer.

The analyzer will work with the `clean_resume` column because it contains the normalized resume text produced during preprocessing.

In [3]:
sample_resume = resume_df.loc[0, "clean_resume"]

print("Sample Resume:")
print("=" * 70)
print(sample_resume[:3000])

Sample Resume:
hr administrator/marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. highlights focused on customer satisfaction team management marketing savvy conflict resolution techniques training and development skilled multi-tasker client relations specialist accomplishments missouri dot supervisor training certification certified by ihg in customer loyalty and marketing by segment hilton worldwide general manager training certification accomplished trainer for cross server hospitality systems such as hilton onq , micros opera pms , fidelio opera reservation system (ors) , holidex completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. ex

## Step 4 — Basic Resume Statistics

The first quality indicators will be measurable characteristics of the resume text.

These include:

- Character count
- Word count
- Sentence count

These measurements will later help identify resumes with unusually little content.

In [4]:
def calculate_resume_statistics(text):
    text = str(text)

    character_count = len(text)
    word_count = len(text.split())
    sentence_count = len(
        re.findall(r"[.!?]+", text)
    )

    return {
        "character_count": character_count,
        "word_count": word_count,
        "sentence_count": sentence_count
    }


sample_stats = calculate_resume_statistics(sample_resume)

print("Resume Statistics")
print("=" * 50)

for key, value in sample_stats.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

Resume Statistics
Character Count: 5027
Word Count: 674
Sentence Count: 26


## Step 5 — Dataset-Level Resume Statistics

The same measurements are calculated for every resume in the dataset.

This allows the system to identify the distribution of resume lengths and content across the complete dataset rather than relying on a single example.

In [5]:
resume_df["quality_character_count"] = (
    resume_df["clean_resume"]
    .fillna("")
    .astype(str)
    .str.len()
)

resume_df["quality_word_count"] = (
    resume_df["clean_resume"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

resume_df["quality_sentence_count"] = (
    resume_df["clean_resume"]
    .fillna("")
    .astype(str)
    .apply(
        lambda text: len(
            re.findall(r"[.!?]+", text)
        )
    )
)

print("Statistics calculated for all resumes.")

print("\nSummary:")
print(
    resume_df[
        [
            "quality_character_count",
            "quality_word_count",
            "quality_sentence_count"
        ]
    ].describe()
)

Statistics calculated for all resumes.

Summary:
       quality_character_count  quality_word_count  quality_sentence_count
count              2484.000000         2484.000000              2484.00000
mean               5921.805556          811.151771                39.15781
std                2681.590763          370.845653                25.93101
min                   0.000000            0.000000                 0.00000
25%                4813.500000          650.750000                22.00000
50%                5537.000000          757.000000                37.00000
75%                6833.500000          933.000000                50.00000
max               35217.000000         5190.000000               407.00000


## Step 6 — Resume Section Detection

A resume can contain useful information even when some common sections are absent.

Therefore, the analyzer will detect the presence of common resume sections rather than requiring every section.

The following section types will be checked:

- Contact Information
- Summary / Objective
- Education
- Experience
- Skills
- Projects
- Certifications

Section detection is based on recognizable keywords and headings in the cleaned resume text.

The detected sections will later be used as evidence for the quality analysis.

In [6]:
section_keywords = {
    "contact_information": [
        "email",
        "phone",
        "mobile",
        "telephone",
        "contact"
    ],

    "summary_objective": [
        "summary",
        "professional summary",
        "career objective",
        "objective",
        "profile"
    ],

    "education": [
        "education",
        "academic",
        "degree",
        "bachelor",
        "master",
        "university",
        "college"
    ],

    "experience": [
        "experience",
        "work experience",
        "professional experience",
        "employment",
        "work history"
    ],

    "skills": [
        "skills",
        "technical skills",
        "professional skills",
        "core skills",
        "competencies"
    ],

    "projects": [
        "projects",
        "project experience",
        "academic projects"
    ],

    "certifications": [
        "certification",
        "certifications",
        "certificate",
        "certified"
    ]
}

print("Section detection rules created successfully.")
print("Number of section types:", len(section_keywords))

Section detection rules created successfully.
Number of section types: 7


## Step 7 — Detect Sections in a Resume

The section detection function searches the normalized resume text for the predefined section keywords.

The result is a dictionary containing `True` or `False` for each section.

This provides an explainable representation of the resume structure.

In [7]:
def detect_resume_sections(text, section_keywords):
    text = str(text).lower()

    detected_sections = {}

    for section, keywords in section_keywords.items():
        detected_sections[section] = any(
            keyword in text
            for keyword in keywords
        )

    return detected_sections

In [8]:
sample_sections = detect_resume_sections(
    sample_resume,
    section_keywords
)

print("Detected Sections")
print("=" * 50)

for section, detected in sample_sections.items():
    print(f"{section}: {detected}")

Detected Sections
contact_information: False
summary_objective: True
education: True
experience: True
skills: True
projects: False
certifications: True


## Step 8 — Analyze Resume Sections Across the Dataset

The section detection function is now applied to all resumes.

This allows us to determine how frequently common resume sections appear in the dataset.

The resulting Boolean columns will also be used later when generating quality indicators.

In [9]:
section_results = resume_df["clean_resume"].apply(
    lambda text: detect_resume_sections(
        text,
        section_keywords
    )
)

section_df = pd.DataFrame(
    section_results.tolist()
)

resume_df = pd.concat(
    [
        resume_df.reset_index(drop=True),
        section_df.reset_index(drop=True)
    ],
    axis=1
)

print("Section detection completed for all resumes.")

print("\nDataset shape:", resume_df.shape)

print("\nSection detection columns:")
print(list(section_df.columns))

Section detection completed for all resumes.

Dataset shape: (2484, 20)

Section detection columns:
['contact_information', 'summary_objective', 'education', 'experience', 'skills', 'projects', 'certifications']


## Step 9 — Section Presence Summary

The following summary shows the number of resumes in which each section was detected.

This is an exploratory validation step. It helps us understand whether the detection rules are producing reasonable results before they are used in the final quality analysis.

In [10]:
section_summary = section_df.sum().sort_values(
    ascending=False
)

print("Resume Section Detection Summary")
print("=" * 60)

for section, count in section_summary.items():
    percentage = (count / len(resume_df)) * 100

    print(
        f"{section}: "
        f"{count} resumes "
        f"({percentage:.2f}%)"
    )

Resume Section Detection Summary
experience: 2483 resumes (99.96%)
education: 2468 resumes (99.36%)
skills: 2459 resumes (98.99%)
summary_objective: 2231 resumes (89.81%)
contact_information: 1253 resumes (50.44%)
certifications: 1176 resumes (47.34%)
projects: 1055 resumes (42.47%)


## Step 10 — Refine Section Detection

The initial section detection used broad keyword matching.

This produced very high detection rates for some sections because section-related words can appear naturally inside other parts of a resume.

For example, the word "degree" may appear in an education description without an explicit Education section heading.

Therefore, the section detection rules will be refined to prioritize common section-heading patterns.

This improves the explainability of the Resume Quality Analyzer and reduces false section detections.

In [11]:
section_heading_patterns = {
    "contact_information": [
        r"\bcontact\b",
        r"\bcontact information\b",
        r"\bpersonal information\b"
    ],

    "summary_objective": [
        r"\bsummary\b",
        r"\bprofessional summary\b",
        r"\bcareer objective\b",
        r"\bobjective\b",
        r"\bprofile\b",
        r"\bprofessional profile\b"
    ],

    "education": [
        r"\beducation\b",
        r"\bacademic background\b",
        r"\bacademic qualifications\b"
    ],

    "experience": [
        r"\bwork experience\b",
        r"\bprofessional experience\b",
        r"\bemployment history\b",
        r"\bwork history\b",
        r"\bexperience\b"
    ],

    "skills": [
        r"\bskills\b",
        r"\btechnical skills\b",
        r"\bprofessional skills\b",
        r"\bcore skills\b",
        r"\bkey skills\b",
        r"\bcompetencies\b"
    ],

    "projects": [
        r"\bprojects\b",
        r"\bproject experience\b",
        r"\bacademic projects\b"
    ],

    "certifications": [
        r"\bcertifications\b",
        r"\bcertification\b",
        r"\bprofessional certifications\b"
    ]
}

print("Refined section-heading rules created.")
print("Number of section types:", len(section_heading_patterns))

Refined section-heading rules created.
Number of section types: 7


## Step 11 — Detect Section Headings

The refined detector searches for common section-heading expressions.

The detector still uses regular expressions, but the rules are more specific than the initial broad keyword approach.

The purpose is not to perfectly understand every resume format. Instead, it provides a transparent and reproducible structural analysis that can later be improved when the application handles uploaded resumes directly.

In [12]:
def detect_section_headings(text, section_patterns):
    text = str(text).lower()

    detected_sections = {}

    for section, patterns in section_patterns.items():

        detected = any(
            re.search(pattern, text)
            for pattern in patterns
        )

        detected_sections[section] = detected

    return detected_sections

In [13]:
refined_sample_sections = detect_section_headings(
    sample_resume,
    section_heading_patterns
)

print("Refined Section Detection")
print("=" * 50)

for section, detected in refined_sample_sections.items():
    print(f"{section}: {detected}")

Refined Section Detection
contact_information: False
summary_objective: True
education: True
experience: True
skills: True
projects: False
certifications: True


In [14]:
refined_section_results = resume_df["clean_resume"].apply(
    lambda text: detect_section_headings(
        text,
        section_heading_patterns
    )
)

refined_section_df = pd.DataFrame(
    refined_section_results.tolist()
)

print("Refined section detection completed.")

Refined section detection completed.


In [15]:
refined_section_summary = (
    refined_section_df
    .sum()
    .sort_values(ascending=False)
)

print("\nRefined Resume Section Detection Summary")
print("=" * 60)

for section, count in refined_section_summary.items():

    percentage = (
        count / len(resume_df)
    ) * 100

    print(
        f"{section}: "
        f"{count} resumes "
        f"({percentage:.2f}%)"
    )


Refined Resume Section Detection Summary
experience: 2482 resumes (99.92%)
skills: 2459 resumes (98.99%)
education: 2439 resumes (98.19%)
summary_objective: 2179 resumes (87.72%)
projects: 1055 resumes (42.47%)
certifications: 667 resumes (26.85%)
contact_information: 522 resumes (21.01%)


## Step 13 — Define Core and Optional Resume Sections

Not every resume requires every possible section.

For the quality analysis, the detected sections are therefore divided into two groups.

### Core Sections

These sections contain information that is generally important for understanding a candidate's background:

- Contact Information
- Education
- Experience
- Skills

### Optional Sections

These sections can strengthen a resume but are not required for every candidate:

- Summary / Objective
- Projects
- Certifications

Missing an optional section will therefore not automatically be considered a quality problem.

The distinction allows the analyzer to provide more meaningful and explainable quality indicators.

In [16]:
core_sections = {
    "contact_information",
    "education",
    "experience",
    "skills"
}

optional_sections = {
    "summary_objective",
    "projects",
    "certifications"
}

print("Core sections:", core_sections)
print("Optional sections:", optional_sections)

Core sections: {'contact_information', 'education', 'skills', 'experience'}
Optional sections: {'projects', 'certifications', 'summary_objective'}


## Step 14 — Core Section Completeness

Each resume is checked for the presence of the four core sections.

The result is represented as a count from 0 to 4.

This is an indicator of structural completeness rather than a judgment of the candidate's ability or qualifications.

In [17]:
resume_df["core_sections_present"] = (
    refined_section_df[
        list(core_sections)
    ]
    .sum(axis=1)
)

print("Core section completeness calculated.")

print("\nDistribution:")
print(
    resume_df["core_sections_present"]
    .value_counts()
    .sort_index()
)

Core section completeness calculated.

Distribution:
core_sections_present
0       1
2      54
3    1922
4     507
Name: count, dtype: int64


## Step 15 — Identify Missing Core Sections

For each resume, the analyzer identifies the core sections that were not detected.

This produces an explainable quality indicator.

For example, instead of simply reporting a low quality score, the system can state:

"Missing contact information"

or

"Missing education and skills sections."

This makes the analyzer more useful to the candidate.

In [18]:
def get_missing_core_sections(row):
    missing = []

    for section in core_sections:
        if not row[section]:
            missing.append(section)

    return sorted(missing)

In [19]:
resume_df["missing_core_sections"] = resume_df.apply(
    get_missing_core_sections,
    axis=1
)

print("Missing core sections identified.")

Missing core sections identified.


In [20]:
for i in range(5):
    print(
        f"Resume {i}:",
        resume_df.loc[i, "missing_core_sections"]
    )

Resume 0: ['contact_information']
Resume 1: ['contact_information']
Resume 2: []
Resume 3: []
Resume 4: ['contact_information']


## Step 16 — Content Completeness

Resume length provides another measurable indicator of content completeness.

The analyzer will use the word count already calculated during the earlier analysis.

Very short or empty resumes can be flagged for further review, while normal-length resumes will not be penalized simply because they are shorter than another resume.

The threshold will be determined from the observed dataset rather than arbitrarily assigning a quality score.

In [21]:
print("Shortest resumes by word count:")
print(
    resume_df[
        ["ID", "Category", "quality_word_count"]
    ]
    .sort_values("quality_word_count")
    .head(15)
)

Shortest resumes by word count:
            ID              Category  quality_word_count
656   12632728  BUSINESS-DEVELOPMENT                   0
1931  22965804          CONSTRUCTION                 113
1049  12696104                 SALES                 122
1102  14381464                 SALES                 128
130   21705160              DESIGNER                 131
153   78149576              DESIGNER                 134
1852  24799301            ACCOUNTANT                 143
1400  35468363                  CHEF                 149
1727  21298336           ENGINEERING                 156
1394  47317494                  CHEF                 164
1039  98108571                 SALES                 169
1026  15273850                 SALES                 169
2033  27187994          CONSTRUCTION                 169
2322  30594821                  ARTS                 172
1390  25953797                  CHEF                 176


In [22]:
print("\nWord-count percentiles:")

print(
    resume_df["quality_word_count"]
    .quantile(
        [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


Word-count percentiles:
0.01     201.66
0.05     302.15
0.10     400.30
0.25     650.75
0.50     757.00
0.75     933.00
0.90    1179.00
0.95    1398.70
0.99    2052.89
Name: quality_word_count, dtype: float64


## Step 17 — Content-Length Indicators

The dataset analysis showed that the 1st percentile of resume word count is approximately 202 words.

Based on this observed distribution, the analyzer will use the following transparent indicators:

- 0 words: Empty resume
- 1–200 words: Very short resume
- More than 200 words: Sufficient textual content

These indicators describe the amount of textual content and do not represent an overall judgment of resume quality.

In [23]:
def classify_content_length(word_count):

    if word_count == 0:
        return "empty"

    elif word_count <= 200:
        return "very_short"

    else:
        return "sufficient"

In [24]:
resume_df["content_length_status"] = (
    resume_df["quality_word_count"]
    .apply(classify_content_length)
)

print("Content-length classification completed.")

print("\nContent Length Distribution:")
print(
    resume_df["content_length_status"]
    .value_counts()
)

Content-length classification completed.

Content Length Distribution:
content_length_status
sufficient    2459
very_short      24
empty            1
Name: count, dtype: int64


## Step 18 — Generate Resume Quality Indicators

The analyzer now combines measurable characteristics into explainable quality indicators.

The indicators include:

- Empty resume
- Very short resume
- Missing core sections
- Detected core sections
- Detected optional sections

These indicators will be used to produce a structured quality report rather than relying on an arbitrary overall score.

In [25]:
def generate_quality_indicators(row):

    indicators = []

    if row["content_length_status"] == "empty":
        indicators.append("Resume contains no textual content.")

    elif row["content_length_status"] == "very_short":
        indicators.append(
            "Resume contains very limited textual content."
        )

    missing_sections = row["missing_core_sections"]

    for section in missing_sections:

        readable_name = section.replace(
            "_", " "
        ).title()

        indicators.append(
            f"Missing core section: {readable_name}."
        )

    if row["core_sections_present"] == len(core_sections):
        indicators.append(
            "All core resume sections were detected."
        )

    return indicators

In [26]:
resume_df["quality_indicators"] = resume_df.apply(
    generate_quality_indicators,
    axis=1
)

print("Quality indicators generated successfully.")

Quality indicators generated successfully.


## Step 19 — Resume Quality Analysis Function

A reusable analysis function is created so that the same quality-analysis logic can later be used by the FastAPI backend.

The function will accept resume text and return structured quality information.

This is important because the final application will analyze individual resumes uploaded by users rather than processing the complete dataset.

In [27]:
def analyze_resume_quality(text):

    text = str(text)

    stats = calculate_resume_statistics(text)

    sections = detect_section_headings(
        text,
        section_heading_patterns
    )

    core_present = sum(
        sections[section]
        for section in core_sections
    )

    missing_core = [
        section
        for section in core_sections
        if not sections[section]
    ]

    content_status = classify_content_length(
        stats["word_count"]
    )

    indicators = []

    if content_status == "empty":
        indicators.append(
            "Resume contains no textual content."
        )

    elif content_status == "very_short":
        indicators.append(
            "Resume contains very limited textual content."
        )

    for section in missing_core:

        readable_name = section.replace(
            "_", " "
        ).title()

        indicators.append(
            f"Missing core section: {readable_name}."
        )

    if core_present == len(core_sections):
        indicators.append(
            "All core resume sections were detected."
        )

    return {
        "character_count": stats["character_count"],
        "word_count": stats["word_count"],
        "sentence_count": stats["sentence_count"],
        "detected_sections": sections,
        "core_sections_present": core_present,
        "missing_core_sections": sorted(missing_core),
        "content_length_status": content_status,
        "quality_indicators": indicators
    }

In [28]:
sample_quality = analyze_resume_quality(
    sample_resume
)

print("=" * 60)
print("RESUME QUALITY ANALYSIS")
print("=" * 60)

print("\nCharacter Count:",
      sample_quality["character_count"])

print("Word Count:",
      sample_quality["word_count"])

print("Sentence Count:",
      sample_quality["sentence_count"])

print("\nCore Sections Present:",
      sample_quality["core_sections_present"])

print("\nMissing Core Sections:")

for section in sample_quality["missing_core_sections"]:
    print("-", section)

print("\nContent Status:",
      sample_quality["content_length_status"])

print("\nQuality Indicators:")

for indicator in sample_quality["quality_indicators"]:
    print("-", indicator)

RESUME QUALITY ANALYSIS

Character Count: 5027
Word Count: 674
Sentence Count: 26

Core Sections Present: 3

Missing Core Sections:
- contact_information

Content Status: sufficient

Quality Indicators:
- Missing core section: Contact Information.


## Step 21 — Validation Using Different Resume Cases

The Resume Quality Analyzer will be tested using resumes with different characteristics.

The validation will include:

1. A normal resume with substantial content
2. A very short or incomplete resume
3. A resume with different structural characteristics

The purpose is to verify that the analyzer produces different and explainable results rather than applying the same output to every resume.

In [29]:
empty_resume = resume_df.loc[
    resume_df["quality_word_count"].idxmin(),
    "clean_resume"
]

empty_quality = analyze_resume_quality(
    empty_resume
)

print("=" * 60)
print("EMPTY RESUME VALIDATION")
print("=" * 60)

print("\nWord Count:",
      empty_quality["word_count"])

print("Content Status:",
      empty_quality["content_length_status"])

print("\nMissing Core Sections:")

for section in empty_quality["missing_core_sections"]:
    print("-", section)

print("\nQuality Indicators:")

for indicator in empty_quality["quality_indicators"]:
    print("-", indicator)

EMPTY RESUME VALIDATION

Word Count: 1
Content Status: very_short

Missing Core Sections:
- contact_information
- education
- experience
- skills

Quality Indicators:
- Resume contains very limited textual content.
- Missing core section: Contact Information.
- Missing core section: Education.
- Missing core section: Skills.
- Missing core section: Experience.


In [30]:
median_word_count = resume_df[
    "quality_word_count"
].median()

median_resume_index = (
    resume_df["quality_word_count"] - median_word_count
).abs().idxmin()

median_resume = resume_df.loc[
    median_resume_index,
    "clean_resume"
]

median_quality = analyze_resume_quality(
    median_resume
)

print("=" * 60)
print("NORMAL RESUME VALIDATION")
print("=" * 60)

print("\nWord Count:",
      median_quality["word_count"])

print("Content Status:",
      median_quality["content_length_status"])

print("\nDetected Core Sections:")

for section in core_sections:

    status = median_quality[
        "detected_sections"
    ][section]

    print(
        f"- {section}: {status}"
    )

print("\nQuality Indicators:")

for indicator in median_quality["quality_indicators"]:
    print("-", indicator)

NORMAL RESUME VALIDATION

Word Count: 757
Content Status: sufficient

Detected Core Sections:
- contact_information: False
- education: True
- skills: True
- experience: True

Quality Indicators:
- Missing core section: Contact Information.


## Step 23 — Batch Validation

The quality analyzer is applied to a sample of resumes from different positions in the dataset.

The validation records:

- Resume ID
- Resume category
- Word count
- Content status
- Number of detected core sections
- Missing core sections

This provides a compact verification that the analyzer works across multiple resumes.

In [31]:
validation_indices = [
    0,
    100,
    500,
    1000,
    1500,
    2000,
    656
]

validation_rows = []

for index in validation_indices:

    text = resume_df.loc[
        index,
        "clean_resume"
    ]

    result = analyze_resume_quality(text)

    validation_rows.append({
        "ID": resume_df.loc[index, "ID"],
        "Category": resume_df.loc[index, "Category"],
        "Word Count": result["word_count"],
        "Content Status": result["content_length_status"],
        "Core Sections": result["core_sections_present"],
        "Missing Core Sections": ", ".join(
            result["missing_core_sections"]
        )
    })

validation_df = pd.DataFrame(
    validation_rows
)

print("Batch validation completed.")

display(validation_df)

Batch validation completed.


,ID,Category,Word Count,Content Status,Core Sections,Missing Core Sections
0,16852973,HR,674,sufficient,3,contact_information
1,10694288,HR,647,sufficient,3,contact_information
2,21614256,ADVOCATE,779,sufficient,4,
3,40987524,SALES,940,sufficient,3,contact_information
4,17775916,FINANCE,903,sufficient,3,contact_information
5,12212468,CONSTRUCTION,1258,sufficient,3,contact_information
6,12632728,BUSINESS-DEVELOPMENT,1,very_short,0,"contact_information, education, experience, sk..."


## Step 24 — Final Structured Resume Quality Result

The final function returns a structured dictionary containing the main quality-analysis information.

This structure will later make it easier for the FastAPI backend and React frontend to display the analysis to the user.

In [32]:
def get_resume_quality_result(text):

    result = analyze_resume_quality(text)

    return {
        "word_count": result["word_count"],
        "sentence_count": result["sentence_count"],
        "character_count": result["character_count"],
        "content_status": result["content_length_status"],
        "detected_sections": result["detected_sections"],
        "core_sections_present": result["core_sections_present"],
        "missing_core_sections": result["missing_core_sections"],
        "quality_indicators": result["quality_indicators"]
    }

In [33]:
final_quality_result = get_resume_quality_result(
    sample_resume
)

print("=" * 60)
print("FINAL RESUME QUALITY RESULT")
print("=" * 60)

for key, value in final_quality_result.items():
    print(f"\n{key}:")
    print(value)

FINAL RESUME QUALITY RESULT

word_count:
674

sentence_count:
26

character_count:
5027

content_status:
sufficient

detected_sections:
{'contact_information': False, 'summary_objective': True, 'education': True, 'experience': True, 'skills': True, 'projects': False, 'certifications': True}

core_sections_present:
3

missing_core_sections:
['contact_information']

quality_indicators:
['Missing core section: Contact Information.']


## Phase 8 — Resume Quality Analyzer: COMPLETE

The Resume Quality Analyzer was implemented and validated using resumes from
the processed resume dataset.

The analyzer evaluates resumes using measurable and explainable indicators
rather than assigning an arbitrary overall quality score.

The implemented analysis includes:

- Character count
- Word count
- Sentence count
- Detection of common resume sections
- Detection of core sections
- Identification of missing core sections
- Content-length classification
- Quality indicators and warnings

The four core sections currently considered are:

- Contact Information
- Education
- Experience
- Skills

The analyzer classifies textual content into:

- Empty
- Very Short
- Sufficient

The system was validated using:

- A very short resume
- A resume near the dataset median word count
- Multiple resumes from different professional categories
- A dataset edge case with extremely limited textual content

The validation demonstrated that the analyzer produces consistent structured
results and correctly identifies missing core sections and limited textual
content.

An important limitation is that section detection is based on textual pattern
matching. Because the cleaned dataset does not preserve the complete visual
layout and formatting of the original resumes, section detection should be
considered an NLP-based approximation rather than a perfect document-layout
analysis.

The final reusable function is:

`get_resume_quality_result(text)`

This function returns a structured result containing the resume statistics,
detected sections, core-section completeness, content status, missing sections,
and quality indicators.

This output can later be integrated into the FastAPI backend.

### Phase 8 Status

- [x] Load processed resume dataset
- [x] Calculate resume text statistics
- [x] Analyze resume content length
- [x] Detect common resume sections
- [x] Identify missing core sections
- [x] Generate quality indicators
- [x] Create reusable quality-analysis function
- [x] Validate very short resume
- [x] Validate normal resume
- [x] Perform batch validation
- [x] Produce structured quality-analysis output

**Phase 8 — Resume Quality Analyzer: COMPLETE**